In [1]:
import pandas as pd
import sys
import os

# Paths
current_dir = os.getcwd()
venv_path = os.path.abspath(os.path.join(current_dir, ".."))
src_path = os.path.join(venv_path, "Scripts", "src")
data_path = os.path.join(venv_path, "Scripts", "data")
model_path = os.path.join(venv_path, "Scripts", "models")

sys.path.insert(0, src_path)

from ml_models import *

In [2]:
df = pd.read_csv(os.path.join(data_path, "processed", "labeled_data.csv"))

print(df.shape)
df.head()

(23171, 26)


,Unnamed: 0,Airline Name,Overall_Rating,Review_Title,Review Date,Verified,Review,Aircraft,Type Of Traveller,Seat Type,...,Inflight Entertainment,Wifi & Connectivity,Value For Money,Recommended,full_review_text,clean_text,review_length,severity_label,risk_flag,severity_id
0,0,AB Aviation,9.0,"""pretty decent airline""",11th November 2019,True,Moroni to Moheli. Turned out to be a pretty ...,NaN,Solo Leisure,Economy Class,...,2.0,1.0,3.0,yes,"""pretty decent airline"" Moroni to Moheli. Tu...",pretty decent airline moroni to moheli turned ...,67,Low,0,0
1,1,AB Aviation,1.0,"""Not a good airline""",25th June 2019,True,Moroni to Anjouan. It is a very small airline...,E120,Solo Leisure,Economy Class,...,2.0,1.0,2.0,no,"""Not a good airline"" Moroni to Anjouan. It is...",not a good airline moroni to anjouan it is a v...,140,Critical,1,3
2,2,AB Aviation,1.0,"""flight was fortunately short""",25th June 2019,True,Anjouan to Dzaoudzi. A very small airline an...,Embraer E120,Solo Leisure,Economy Class,...,2.0,1.0,2.0,no,"""flight was fortunately short"" Anjouan to Dz...",flight was fortunately short anjouan to dzaoud...,71,Critical,1,3
3,3,Adria Airways,1.0,"""I will never fly again with Adria""",28th September 2019,False,Please do a favor yourself and do not fly wi...,NaN,Solo Leisure,Economy Class,...,2.0,1.0,1.0,no,"""I will never fly again with Adria"" Please d...",i will never fly again with adria please do a ...,123,Critical,1,3
4,4,Adria Airways,1.0,"""it ruined our last days of holidays""",24th September 2019,True,Do not book a flight with this airline! My fr...,NaN,Couple Leisure,Economy Class,...,1.0,1.0,1.0,no,"""it ruined our last days of holidays"" Do not ...",it ruined our last days of holidays do not boo...,117,Critical,1,3


In [3]:
text, numeric = prepare_features(df)

X_text, tfidf = tfidf_features(text)
X_combined = combine_features(X_text, numeric)

y_severity = df["severity_id"]
y_risk = df["risk_flag"]

In [4]:
from sklearn.model_selection import train_test_split

X_text_train, X_text_test, X_comb_train, X_comb_test, y_train_sev, y_test_sev = train_test_split(
    X_text, X_combined, y_severity, test_size=0.2, random_state=42
)

X_comb_train_r, X_comb_test_r, y_train_risk, y_test_risk = train_test_split(
    X_combined, y_risk, test_size=0.2, random_state=42
)

In [5]:
import importlib
import ml_models
importlib.reload(ml_models)

<module 'ml_models' from 'd:\\AI-Based Airline Complaint Severity Detection & Service Risk Classification System\\venv\\Scripts\\src\\ml_models.py'>

In [6]:
print("Any NaN in numeric:", numeric.isnull().sum())
print("Any NaN in text:", text.isnull().sum())

Any NaN in numeric: overall_rating            0
seat_comfort              0
cabin_staff_service       0
food_and_beverages        0
ground_service            0
inflight_entertainment    0
wifi_and_connectivity     0
value_for_money           0
review_length             0
dtype: int64
Any NaN in text: 0


In [7]:
import numpy as np
print("Any NaN in combined:", np.isnan(X_combined.data).sum())

Any NaN in combined: 0


In [8]:
import inspect
from ml_models import train_severity_models
print(inspect.signature(train_severity_models))

(X_train_text, X_train_combined, y_train)


In [9]:
severity_models = train_severity_models(X_text_train, X_comb_train, y_train_sev)
risk_models = train_risk_models(X_comb_train_r, y_train_risk)

In [10]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
from sklearn.metrics import classification_report

for name, model in severity_models.items():
    if name == "Naive Bayes":
        preds = model.predict(X_text_test)
    else:
        preds = model.predict(X_comb_test)

    print("Severity Model:", name)
    print(classification_report(y_test_sev, preds))

Severity Model: Logistic Regression
              precision    recall  f1-score   support

           0       0.84      0.97      0.90       764
           1       0.76      0.19      0.31       227
           2       0.25      0.02      0.04       250
           3       0.89      0.97      0.93      3394

    accuracy                           0.88      4635
   macro avg       0.69      0.54      0.55      4635
weighted avg       0.84      0.88      0.85      4635

Severity Model: Linear SVM
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       764
           1       0.61      0.20      0.30       227
           2       0.38      0.11      0.17       250
           3       0.92      0.97      0.94      3394

    accuracy                           0.89      4635
   macro avg       0.69      0.57      0.58      4635
weighted avg       0.86      0.89      0.86      4635

Severity Model: Naive Bayes
              precision    recall  f1-s

In [12]:
for name, model in risk_models.items():
    preds = model.predict(X_comb_test_r)
    print("Risk Model:", name)
    print(classification_report(y_test_risk, preds))

Risk Model: Logistic Regression
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       877
           1       0.97      0.96      0.97      3758

    accuracy                           0.95      4635
   macro avg       0.91      0.93      0.92      4635
weighted avg       0.95      0.95      0.95      4635

Risk Model: Random Forest
              precision    recall  f1-score   support

           0       0.95      0.79      0.86       877
           1       0.95      0.99      0.97      3758

    accuracy                           0.95      4635
   macro avg       0.95      0.89      0.92      4635
weighted avg       0.95      0.95      0.95      4635



In [13]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

# Severity models
for name, model in severity_models.items():
    if name == "Naive Bayes":
        preds = model.predict(X_text_test)
    else:
        preds = model.predict(X_comb_test)

    results.append([
        name,
        "Severity",
        accuracy_score(y_test_sev, preds),
        precision_score(y_test_sev, preds, average="weighted", zero_division=0),
        recall_score(y_test_sev, preds, average="weighted", zero_division=0),
        f1_score(y_test_sev, preds, average="weighted", zero_division=0)
    ])

# Risk models
for name, model in risk_models.items():
    preds = model.predict(X_comb_test_r)

    results.append([
        name,
        "Risk",
        accuracy_score(y_test_risk, preds),
        precision_score(y_test_risk, preds, zero_division=0),
        recall_score(y_test_risk, preds, zero_division=0),
        f1_score(y_test_risk, preds, zero_division=0)
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Task", "Accuracy", "Precision", "Recall", "F1"]
)

results_df

,Model,Task,Accuracy,Precision,Recall,F1
0,Logistic Regression,Severity,0.878533,0.843647,0.878533,0.846034
1,Linear SVM,Severity,0.889536,0.860617,0.889536,0.864901
2,Naive Bayes,Severity,0.769795,0.711775,0.769795,0.736270
3,Logistic Regression,Risk,0.949946,0.973416,0.964609,0.968992
4,Random Forest,Risk,0.951888,0.951930,0.990687,0.970922


In [14]:
results_df.to_csv("model_comparison.csv", index=False)

In [103]:
import joblib
import os

model_path = os.path.join(venv_path, "Scripts", "models")

joblib.dump(tfidf, os.path.join(model_path, "tfidf.pkl"))
joblib.dump(severity_models["Logistic Regression"], os.path.join(model_path, "severity_model.pkl"))
joblib.dump(risk_models["Logistic Regression"], os.path.join(model_path, "risk_model.pkl"))

print("Models saved successfully")

Models saved successfully


In [15]:
import joblib
import os

models_path = os.path.join(venv_path, "Scripts", "models")

severity_path = os.path.join(models_path, "severity")
risk_path = os.path.join(models_path, "risk")
vectorizer_path = os.path.join(models_path, "vectorizers")

os.makedirs(severity_path, exist_ok=True)
os.makedirs(risk_path, exist_ok=True)
os.makedirs(vectorizer_path, exist_ok=True)

# Save vectorizer
joblib.dump(tfidf, os.path.join(vectorizer_path, "tfidf.pkl"))

# Save models
joblib.dump(severity_models["Logistic Regression"], os.path.join(severity_path, "severity_model.pkl"))
joblib.dump(risk_models["Logistic Regression"], os.path.join(risk_path, "risk_model.pkl"))

print("Models and vectorizer saved successfully")

Models and vectorizer saved successfully


In [16]:
import joblib
import os

severity_path = "Scripts/models/severity"
risk_path = "Scripts/models/risk"

os.makedirs(severity_path, exist_ok=True)
os.makedirs(risk_path, exist_ok=True)

# Save severity models
for name, model in severity_models.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, os.path.join(severity_path, filename))

# Save risk models
for name, model in risk_models.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, os.path.join(risk_path, filename))

print("ML models saved successfully")

ML models saved successfully


In [17]:
import joblib
import os

severity_path = "Scripts/models/severity"
risk_path = "Scripts/models/risk"
vectorizer_path = "Scripts/models/vectorizers"

os.makedirs(severity_path, exist_ok=True)
os.makedirs(risk_path, exist_ok=True)
os.makedirs(vectorizer_path, exist_ok=True)

# Save severity models
for name, model in severity_models.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, os.path.join(severity_path, filename))

# Save risk models
for name, model in risk_models.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, os.path.join(risk_path, filename))

# Save TF-IDF
joblib.dump(tfidf, os.path.join(vectorizer_path, "tfidf.pkl"))

print("All ML models saved successfully")

All ML models saved successfully


In [ ]:
import joblib
joblib.dump(tfidf, "Scripts/models/vectorizers/tfidf.pkl")
print("TF-IDF saved")